In [29]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown

from anthropic import Anthropic



In [2]:
load_dotenv(override=True)

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

client = Anthropic(api_key=anthropic_api_key)

model = "claude-sonnet-4-5-20250929"

In [16]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [6]:
# Stop sequences

message = client.messages.create(
    model=model,
    max_tokens=1000,
    stop_sequences=["END"], # Stop generating at double new line
    messages=[{"role": "user", "content": "Write a short poem"}]
)

print(message.content[0].text)

# A Moment

The sky holds its breath at dawn,
Colors bleeding through the seam
Where night meets day—a fragile line
Between what was and what might be.


In [8]:
# Top k (limits sample to the k most likely tokens), and Top P (uses nucleus sampling with cumulative probability)

message = client.messages.create(
    model=model,
    max_tokens=1000,
    temperature=0.8,
    stop_sequences=["END"], # Stop generating at double new line
    messages=[{"role": "user", "content": "Write a short poem"}]
)

print(message.content[0].text)

# Fleeting Moment

A leaf drifts down on autumn air,
Golden, spinning without care.
One breath of wind, then it is gone—
The tree stands bare, the season moves on.


In [19]:
messages = []

add_user_message(messages, "Generate a very short AWS Eventbridge Rule as JSON.")

text = chat(messages)
print(text)

```json
{
  "Source": ["aws.ec2"],
  "DetailType": ["EC2 Instance State-change Notification"],
  "Detail": {
    "state": ["running"]
  }
}
```


In [ ]:
# Assistant message prefill - assistant turn with "```json", stop generating once it gets to the closing "```"

messages = []

add_user_message(messages, "Generate a very short AWS Eventbridge Rule as JSON.")

add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

print(text)


{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}



In [32]:
messages = []

prompt = """Generate three difference sample AWS CLI commands.  Each should be very short."""

add_user_message(messages, prompt)

add_assistant_message(messages, "```bash")

text = chat(messages, stop_sequences=["```"])
text.strip()

Markdown(text)


aws s3 ls

aws ec2 describe-instances --region us-east-1

aws iam list-users
